In [1]:
!python --version

import torch
print(torch.__version__)
print(torch.cuda.is_available())

!pip install -U transformers
!pip install -U datasets
!pip install tensorboard
!pip install sentencepiece
!pip install accelerate
!pip install evaluate==0.4.0
!pip install rouge_score
!pip install bleu
!pip install -U scikit-learn
!pip install nltk datasets
!pip install sacrebleu

Python 3.10.15
2.5.1+cu118
True
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
import torch
import numpy as np
import nltk
import evaluate
import sacrebleu
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

os.environ["CUDA_VISIBLE_DEVICES"]="1,3"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:32"

torch.cuda.empty_cache()


In [3]:
import json
lista_pictogramas_ids=[]
with open("ids_arasaac.txt", 'r', encoding='utf-8') as archivo:
  lineas = archivo.readlines()
  for linea in lineas:
    lista_pictogramas_ids.append(linea.strip())

print(len(lista_pictogramas_ids))

38828


In [4]:
dataset_train = load_dataset('json', data_files='train_data.json')['train']
dataset_test = load_dataset('json', data_files='test_data.json')['train']
dataset_valid = load_dataset('json', data_files='validation_data.json')['train']

# Mostrar estadísticas básicas
print(dataset_train)
print(dataset_test)
print(dataset_valid)

Dataset({
    features: ['id', 'oracion', 'traduccion'],
    num_rows: 35131
})
Dataset({
    features: ['id', 'oracion', 'traduccion'],
    num_rows: 423
})
Dataset({
    features: ['id', 'oracion', 'traduccion'],
    num_rows: 422
})


In [5]:
model_checkpoint = "flax-community/spanish-t5-small" #"vgaraujov/t5-base-spanish"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params:,} total parameters.")

total_trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad)
print(f"{total_trainable_params:,} training parameters.")

# Agregar nuevos tokens al tokenizer
existing_tokens = set(tokenizer.get_vocab().keys())
new_tokens = [token for token in lista_pictogramas_ids if token not in existing_tokens]
tokenizer.add_tokens(new_tokens)

model.resize_token_embeddings(len(tokenizer))

# Validar parámetros actualizados
print("After adding new tokens:")
total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params:,} total parameters.")
total_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{total_trainable_params:,} training parameters.")

new_token_ids = tokenizer.convert_tokens_to_ids(new_tokens)
print(f"Sample IDs of new tokens: {new_token_ids[:10]}")

new_embeddings = model.get_input_embeddings().weight.data[new_token_ids]
print("Sample embeddings for new tokens:", new_embeddings[:5])

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Convertir los nuevos tokens a sus IDs
new_token_ids = tokenizer.convert_tokens_to_ids(new_tokens)

# Mostrar ejemplos de tokens agregados y sus IDs
for token, token_id in zip(new_tokens[:10], new_token_ids[:10]):
    print(f"Token: {token}, ID: {token_id}")

60,493,824 total parameters.
60,493,824 training parameters.


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


After adding new tokens:
80,367,104 total parameters.
80,367,104 training parameters.
Sample IDs of new tokens: [32103, 32104, 32105, 32106, 32107, 32108, 32109, 32110, 32111, 32112]
Sample embeddings for new tokens: tensor([[ 0.4681, -0.5677, -0.4633,  ...,  0.7039, -0.5305, -0.0855],
        [ 0.4681, -0.5677, -0.4633,  ...,  0.7039, -0.5305, -0.0855],
        [ 0.4681, -0.5677, -0.4633,  ...,  0.7039, -0.5305, -0.0855],
        [ 0.4681, -0.5677, -0.4633,  ...,  0.7039, -0.5305, -0.0855],
        [ 0.4681, -0.5677, -0.4633,  ...,  0.7039, -0.5305, -0.0855]])
Token: pict_2247_acera, ID: 32103
Token: pict_2247_acera_plural, ID: 32104
Token: pict_2247_vereda, ID: 32105
Token: pict_2247_vereda_plural, ID: 32106
Token: pict_2248_agua, ID: 32107
Token: pict_2248_agua_plural, ID: 32108
Token: pict_2249_alfombra, ID: 32109
Token: pict_2249_alfombra_plural, ID: 32110
Token: pict_2250_almohada, ID: 32111
Token: pict_2250_almohada_plural, ID: 32112


In [6]:
model.to(torch.device('cuda'))
model.device

device(type='cuda', index=0)

In [7]:
max_input_length = 20
max_target_length = 20

batch_size=16
metric ="bleu"
model_name = "t5-xgen-finetuned"
evaluation_strategy = "epoch"
save_strategy="epoch"
overwrite_output_dir=True
learning_rate=10e-5
gradient_accumulation_steps=1
weight_decay=0.01
do_train=True
do_eval=True
save_total_limit=20
num_train_epochs=20
seed=42
predict_with_generate=True
fp16=True
metric_for_best_model="bleu"
load_best_model_at_end=True
generation_max_length = max_target_length
logging_strategy="epoch"
eval_accumulation_steps=2

In [8]:
def preprocess_function(examples):
    inputs = ["translate: " + oracion for oracion in examples['oracion']]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")    
    labels = tokenizer(text_target=examples["traduccion"], max_length=max_target_length, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [9]:
encoded_train = dataset_train.map(preprocess_function, batched=True)
encoded_test = dataset_test.map(preprocess_function, batched=True)
encoded_validation = dataset_valid.map(preprocess_function, batched=True)

Map:   0%|          | 0/423 [00:00<?, ? examples/s]

In [10]:
nltk.download("punkt", quiet=True)

bleu_metric = evaluate.load("bleu")
chrf_metric = evaluate.load('chrf')

def compute_metrics_v2(eval_preds):
    print(f"metrics v2")
    predictions, labels = eval_preds

    # Reemplazar -100 con pad_token_id tanto en predicciones como en etiquetas
    pad_token_id = tokenizer.pad_token_id
    predictions = np.where(predictions != -100, predictions, pad_token_id)
    labels = np.where(labels != -100, labels, pad_token_id)
    
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
     
    # Asegurarse de que BLEU reciba cadenas completas, no listas de palabras
    preds = [" ".join(pred.split()) for pred in decoded_preds]
    refs = [[" ".join(label.split())] for label in decoded_labels]  # Lista de listas para referencias
    
    print(f"bleu_preds {preds}")
    print(f"bleu_refs {refs}")
    
    # Calcular BLEU
    try:
        bleu_result = bleu_metric.compute(predictions=preds, references=refs)
        bleu_score = {"bleu": bleu_result["bleu"], "precisions": bleu_result["precisions"]}
    except Exception as e:
        print(f"Error al calcular BLEU: {e}")
        bleu_score = {"bleu": 0.0}

    # Calcular CHRF++
    try:
        chrf_result = chrf_metric.compute(predictions=preds, references=refs)
        print(chrf_result)
        chrf_score = {"chrf": chrf_result["score"]}
    except Exception as e:
        print(f"Error al calcular CHRF++: {e}")
        chrf_score = {"chrf": 0.0}
        
    # Combinar resultados
    combined_results = {**bleu_score, **chrf_score}
    return combined_results

In [11]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy = evaluation_strategy,
    save_strategy = save_strategy,
    overwrite_output_dir = overwrite_output_dir,
    learning_rate = learning_rate,
    per_device_train_batch_size = batch_size,
    per_device_eval_batch_size= batch_size,
    gradient_accumulation_steps= gradient_accumulation_steps,
    weight_decay= weight_decay,
    do_train= do_train,
    do_eval= do_eval,
    save_total_limit= save_total_limit,
    num_train_epochs= num_train_epochs,
    seed= seed,
    predict_with_generate= predict_with_generate,
    fp16= fp16,
    generation_max_length=generation_max_length,
    logging_strategy=logging_strategy,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    )

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=encoded_train,
    eval_dataset=encoded_test,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_v2
)



C:\ProgramData\anaconda3\envs\thesis\lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\apare\AppData\Local\Temp\ipykernel_12888\3537574832.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [12]:
import numpy as np

import nltk
nltk.download('punkt')

trainer.train()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\apare\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Bleu,Precisions,Chrf
1,1.242200,1.468884,0.000000,"[0.4663292562912976, 0.1546853982962188, 0.005239758653540806, 0.0]",23.394625
2,0.885400,1.425950,0.070246,"[0.4954918032786885, 0.17799681113204813, 0.019450447669033654, 0.01419376134675689]",26.750958
3,0.815000,1.411914,0.083427,"[0.48839071257005606, 0.18405428329092452, 0.028249436513899325, 0.01907662712407823]",27.362854
4,0.760000,1.439347,0.112207,"[0.5149515653516776, 0.2064179104477612, 0.04777070063694268, 0.031218014329580348]",30.430247
5,0.714100,1.384178,0.150611,"[0.5239978183801473, 0.23006800752423673, 0.07643704731083371, 0.05583923571075605]",33.004849
6,0.664400,1.435089,0.217911,"[0.543251658318668, 0.27843193566915564, 0.13657195233730524, 0.10915320606950563]",36.731241
7,0.616800,1.410888,0.238522,"[0.5578684429641965, 0.29765590446705, 0.15601946319259144, 0.12493711219184973]",38.811786
8,0.577400,1.427992,0.264257,"[0.5614748542768063, 0.3183779119930975, 0.1837077793061287, 0.1484922575387123]",41.122940
9,0.546400,1.464642,0.298683,"[0.5840559440559441, 0.3485952133194589, 0.21743946827029592, 0.17977338068662269]",43.639567
10,0.520500,1.461255,0.315683,"[0.5902690645476091, 0.3632592592592593, 0.23624034064027755, 0.19605861546235473]",45.788660


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

metrics v2
bleu_preds ['pict_3418_?', 'pict_3418_? pict_3418_? pict_3418_? pict_3418_? pict_3418_? pict_3418_? pict_3418_?', 'pict_3418_? pict_3418_? pict_3418_? pict_3418_?', 'pict_3418_? pict_3418_? pict_3418_? pict_3418_?', 'pict_3418_? pict_3418_? pict_3418_? pict_3418_? pict_3418_? pict_3418_?', 'pict_3418_? pict_3418_? pict_3418_? ambiente', 'pict_3418_? pict_3418_? pict_3418_? pict_3418_?', 'pict_3418_? pict_3418_? pict_3418_? pict_3418_? pict_3418_?', 'pict_3418_? pict_3418_? pict_3418_?', 'pict_3418_? pict_3418_? pict_3418_? pict_3418_?', 'pict_3418_? pict_3418_? pict_3418_?', 'pict_3418_? pict_3418_? pict_3418_?', 'pict_3418_? pict_3418_? pict_3418_? pict_3418_? pict_3418_? pict_3418_? pict_3418_?', 'pict_3418_? pict_3418_? idea', 'pict_3418_? pict_3418_? pict_3418_? pict_3418_?', 'pict_3418_? pict_3418_? glóbulo pict_3418_?', 'pict_3418_? pict_3418_? pict_3418_? geológico', 'pict_3418_? pict_3418_? pict_3418_? pict_3418_?', 'pict_3418_? pict_3418_? pict_3418_? pict_3418_?', 

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

...

{'score': 52.2842736591607, 'char_order': 6, 'word_order': 0, 'beta': 2}


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

metrics v2
bleu_preds ['luna pict_26176_completa', 'pict_3047_y borro pict_7029_la pict_3114_hoja pict_7074_de pict_8349_papel', 'pict_3414_¡ pict_12268_nuestro pict_9870_presentación pict_3417_!', 'pict_3415_¿ pict_7764_dónde pict_5466_estar pict_3418_?', 'pict_11749_hacer pict_8474_un pict_2359_cuaderno pict_7074_de pict_12266_mis pict_8008_actividades', 'pict_11691_cuidar pict_8476_el pict_23731_aire', 'pict_12281_tu pict_7022_día pict_7074_de pict_6483_elegir', 'pict_6975_niños pict_7034_en pict_36147_cerámica pict_7184_nos pict_8029_aprender', 'pict_2474_observar pict_7029_la pict_2248_agua', 'pict_3415_¿ pict_7764_dónde pict_2380_escribir pict_3418_?', 'pict_11705_mejorar pict_3047_y pict_11691_cuidar', 'pict_2474_observar pict_5466_estar pict_5358_final', 'pict_2381_escuchar pict_7029_la pict_9870_exposición pict_7074_de pict_7033_los pict_7062_compañeros', 'pict_8579_explicar pict_8474_una idea', 'pict_3415_¿ pict_7212_por pict_22624_qué pict_3418_?', 'pict_2474_observar pict_8

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=43920, training_loss=0.5860729933219312, metrics={'train_runtime': 29194.1149, 'train_samples_per_second': 24.067, 'train_steps_per_second': 1.504, 'total_flos': 3714603771494400.0, 'train_loss': 0.5860729933219312, 'epoch': 20.0})

In [ ]:
## Se necesita saber con exactitud como se llama el folder

tokenizerv1 = AutoTokenizer.from_pretrained('./results/checkpoint-2196')
modelv1 = AutoModelForSeq2SeqLM.from_pretrained('./results/checkpoint-2196')

device = torch.device("cuda")
modelv1.to(device)
modelv1.eval()

In [ ]:
# Función para preparar los datos para la predicción
def prepare_data_for_prediction(dataset, tokenizer):
    # Transformar los datos para que sean aptos para el modelo
    prepared_data = []
    for example in dataset:
        inputs = tokenizer("translate: " + example['oracion'], return_tensors="pt", padding="max_length", truncation=True, max_length=max_input_length)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        prepared_data.append(inputs)
    return prepared_data

# Preparar los datos de validación para la predicción
prepared_validation_data = prepare_data_for_prediction(dataset_valid, tokenizerv1)

# Lista para guardar las predicciones
predictions = []

# Generar predicciones para cada entrada en los datos de validación
for inputs in prepared_validation_data:
    with torch.no_grad():
        outputs = modelv1.generate(**inputs, max_length=max_target_length)
    decoded_output = tokenizerv1.decode(outputs[0], skip_special_tokens=True)
    predictions.append(decoded_output)

# Imprimir las predicciones
for i, prediction in enumerate(predictions):
    print(f"Prediction {i+1}: {prediction}")

In [ ]:
## Pendiente de hacer pruebas

oracion_traducida = " ".join(traduccion).strip()
oracion_traducida

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def obtener_tokens(oracion):
    resultado = []
    oracion_split = oracion.strip().split(" ")
    n = len(oracion_split)
    print(n)
    
    i = 0
    while i < n:
        elemento = oracion_split[i].strip()
        if elemento.startswith("#"):
            if elemento.endswith("#"):
                resultado.append(elemento)
                i = i + 1
            else:
                j = i + 1
                tmp = elemento
                while j < n and not oracion_split[j].endswith("#"):
                    tmp = f"{tmp} {oracion_split[j].strip()}"
                    j = j + 1
                tmp = f"{tmp} {oracion_split[j].strip()}"
                i = j + 1
                resultado.append(tmp) 
        else:
            tmp = elemento
            j = i + 1
            while j < n and not oracion_split[j].startswith("#"):
                tmp = f"{tmp} {oracion_split[j].strip()}"
                j = j + 1
            i = j
            resultado.append(tmp) 
    return resultado
    
def obtener_ruta(elemento):
  if not elemento.startswith("#"):
      return elemento
      
  valores = [elem for elem in elemento.split('#') if elem]
    
  try:
    id = int(valores[0])
  except ValueError:
    return valores[0]

  if len(valores) == 2:
    return f"imagenes_procesadas/{id}.png"

  pos = valores[2]
  return f"imagenes_procesadas/{id}_{pos}.png"

def imprimir_imagenes(rutas_imagenes):
  num_imagenes = len(rutas_imagenes)

  # Crear subplots
  fig, axes = plt.subplots(1, num_imagenes, figsize=(20, 20))
  if num_imagenes == 1:
      axes = [axes]
  for ax, ruta_imagen in zip(axes, rutas_imagenes):
      if ruta_imagen.startswith("imagenes_procesadas"):
        img = mpimg.imread(ruta_imagen)
        ax.imshow(img)
        ax.axis('off')
      else:
        ax.axis('off')  # Ocultar los ejes
        ax.text(0.5, 0.5, ruta_imagen, fontsize=12, ha='center', va='center')
  plt.tight_layout()
  plt.show()

In [ ]:
tokens = obtener_tokens(oracion_traducida) 

rutas_imagenes = []
for elemento in tokens:
  rutas_imagenes.append(obtener_ruta(elemento))

print(f"oracion original: {dataset_valid[idx].get('oracion')}")
print(f"traduccion modelo base: {dataset_valid[idx].get('traduccion')}")
print(f"traduccion modelo T5: {oracion_traducida}")
imprimir_imagenes(rutas_imagenes)


In [ ]:
dataset_valid01 = load_dataset(
    'json',
    data_files='pruebasUnitarias.json',
    split='train'
)
encoded_val_user = dataset_valid01.map(preprocess_function, batched=True)

predict01=trainer.predict(encoded_val_user,max_length=max_target_length)


In [ ]:
idx00 = 0
prediccionT5 = predict01.predictions[idx00]
prediccionT5 = np.where(prediccionT5 != -100, prediccionT5, tokenizer.pad_token_id)

traducciont5 = tokenizer.batch_decode(
        prediccionT5,
        skip_special_tokens=True
    )
print(traducciont5)
oracion_traducida_t5 = " ".join(traducciont5).strip()

oracion_traducida_t5

In [ ]:
tokenst5 = obtener_tokens(oracion_traducida_t5) 

rutas_imagenes01 = []
for elemento in tokenst5:
  rutas_imagenes01.append(obtener_ruta(elemento))


rutas_imagenes02 = []
tokensValUsuario = obtener_tokens(dataset_valid01[idx00].get('traduccion')) 

for elemento in tokensValUsuario:
  rutas_imagenes02.append(obtener_ruta(elemento))
    
print(f"oracion: {dataset_valid01[idx00].get('oracion')}")
print(f"traducción validada: {dataset_valid01[idx00].get('traduccion')}")
print(f"traducción modelo T5: {oracion_traducida_t5}")

imprimir_imagenes(rutas_imagenes02)
imprimir_imagenes(rutas_imagenes01)



In [ ]:
dataset_valid02 = load_dataset(
    'json',
    data_files='pruebasUnitarias.json',
    split='train'
)
encoded_val_user = dataset_valid02.map(preprocess_function, batched=True)

predict02=trainer.predict(encoded_val_user,max_length=max_target_length)

idx00 = 0
prediccionT501 = predict02.predictions[idx00]
prediccionT501 = np.where(prediccionT501 != -100, prediccionT501, tokenizer.pad_token_id)

traducciont501 = tokenizer.batch_decode(
        prediccionT501,
        skip_special_tokens=True
    )
print(traducciont501)
oracion_traducida_t501 = " ".join(traducciont501).strip()

tokenst501 = obtener_tokens(oracion_traducida_t501) 

rutas_imagenes03 = []
for elemento in tokenst501:
  rutas_imagenes03.append(obtener_ruta(elemento))

print(f"oracion: {dataset_valid02[idx00].get('oracion')}")
print(f"traducción modelo T5: {oracion_traducida_t501}")

imprimir_imagenes(rutas_imagenes03)